In [ ]:
#Cell 1
!pip install evaluate
import torch
import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import openai
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# HF Token (for LLaMA-3)
HF_TOKEN = "your_hf_token_here"
openai.api_key = "your_openai_key_here"  # For GPT fallback

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
import json as _json

from datasets import load_dataset
# Load datasets and normalize ground-truth answers
truthfulqa_raw = load_dataset('truthfulqa/truthful_qa', 'generation', split='validation')
popqa_raw = load_dataset('akariasai/PopQA', split='test[:1000]')
# # TruthfulQA
# truthfulqa = load_dataset("truthful_qa", "multiple_choice", split="validation")

def get_truthfulqa_answer(example):
    #labels = example['mc1_targets']['labels']
    #choices = example['mc1_targets']['choices']
    #correct_idx = labels.index(1)
    return {
        'question': example['question'],
        'best_answer': example['correct_answers'],  # now a list
    }

truthfulqa = truthfulqa_raw.map(get_truthfulqa_answer, remove_columns=truthfulqa_raw.column_names)

def get_popqa_answer(example):
    raw = example.get('possible_answers', '[]')
    # possible_answers is a JSON string, e.g. '["politician", "lawyer"]'
    try:
        answers = _json.loads(raw) if isinstance(raw, str) else raw
    except Exception:
        answers = []
    best_answer = answers[0] if answers else ''
    return {
        'question': example['question'],
        'best_answer': best_answer,
        's_pop': example.get('s_pop', None),
        'o_pop': example.get('o_pop', None),
    }

popqa = popqa_raw.map(get_popqa_answer, remove_columns=popqa_raw.column_names)

def assign_popularity_tier(example):
    vals = [v for v in [example.get('s_pop'), example.get('o_pop')] if v is not None]
    avg_pop = sum(vals)/len(vals) if vals else 0
    if avg_pop < 1000:
        tier = 'low'
    elif avg_pop < 100000:
        tier = 'mid'
    else:
        tier = 'high'
    return {'popularity_tier': tier, 'avg_popularity': avg_pop}

popqa = popqa.map(assign_popularity_tier)

datasets = {
    'TruthfulQA': truthfulqa.select(range(50)),
    'PopQA': popqa.select(range(50)),
}

print('Datasets loaded.')
for name, ds in datasets.items():
    print(name, len(ds))
    print(ds[0])
    print('-'*60)


In [ ]:
print(truthfulqa[0])

In [ ]:
!huggingface-cli login

In [ ]:
!hf models ls --search "meta-llama/Meta-Llama-3-8B"

In [ ]:
#Cell 3
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# 1. Get your token from https://huggingface.co/settings/tokens (Read access is enough)
HF_TOKEN = "hf0000000"  # Replace with your real token

# 2. Login (one time)
login(token=HF_TOKEN)

# 3. Load model/tokenizer
# model_name = "meta-llama/Meta-Llama-3-8B"
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN,  # This is the correct parameter name
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16, # Changed from torch_dtype to dtype
    device_map="auto",
    token=HF_TOKEN
)

print("Llama-3-8B loaded successfully!")

In [ ]:
def generate_response(prompt, max_new_tokens=100, num_samples=1, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    responses = []
    with torch.no_grad():
        for _ in range(num_samples):
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True, pad_token_id=tokenizer.eos_token_id)
            response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            responses.append(response.strip())
    return responses[0] if num_samples == 1 else responses

prompt_templates = {
    "baseline": "Question: {question}\nAnswer:",
    "cot": "Question: {question}\nLet's think step by step.\nAnswer:",
    "self_consistency": lambda q: [f"Question: {q}\nLet's think step by step.\nAnswer:"]  # 5 samples later
}

In [ ]:
import re

def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def soft_match(prediction: str, reference: str) -> bool:
    # Handle both single string and list of correct answers
    if isinstance(reference, list):
        return any(soft_match(prediction, r) for r in reference)

    pred_norm = normalize(prediction)
    ref_norm  = normalize(reference)
    if ref_norm in pred_norm or pred_norm in ref_norm:
        return True
    tokens = [t for t in min([ref_norm, pred_norm], key=len).split() if len(t) > 2]
    if tokens and all(tok in max([ref_norm, pred_norm], key=len) for tok in tokens):
        return True
    return False

def compute_metrics(predictions: list, references: list, raw_outputs: list = None) -> dict:
    assert len(predictions) == len(references)
    correct = 0
    for i, (pred, ref) in enumerate(zip(predictions, references)):
        if isinstance(ref, list) and raw_outputs is not None:
            if soft_match(raw_outputs[i], ref) or soft_match(pred, ref):
                correct += 1
        else:
            if soft_match(pred, ref):
                correct += 1
    accuracy = correct / len(predictions) if predictions else 0.0
    return {
        "accuracy": round(accuracy, 4),
        "hallucination_rate": round(1 - accuracy, 4),
        "n": len(predictions),
    }

print("compute_metrics ready.")

In [ ]:
# ── CELL: extract_final_answer ───────────────────────────────
# Place this BEFORE the tier-loop cell and the main results loop.

import re

def extract_final_answer(raw_text: str) -> str:
    raw_text = raw_text.strip()

    # 1. Take the first sentence before any fake Q&A continuation
    first_sentence = re.split(r"[.\n]", raw_text)[0].strip()

    # 2. Extract predicate from "X is a Y" → "Y"
    match = re.search(
        r"\b(?:is|was|are|were)\s+(?:a|an|the)?\s*(.+)",
        first_sentence, re.IGNORECASE
    )
    if match:
        return match.group(1).strip().lower()

    # 3. If there's an explicit Answer: marker (CoT), use that instead
    if "answer:" in raw_text.lower():
        parts = re.split(r"[Aa]nswer\s*:", raw_text)
        candidate = parts[1].strip()
        first = re.split(r"[.\n]", candidate)[0].strip()
        if first:
            return first.lower()

    return first_sentence.lower()

In [ ]:
# DEBUG: inspect one raw prediction vs reference
example = list(datasets['PopQA'])[0]
question = example['question']
ref = example['best_answer']

raw = generate_response(prompt_templates['baseline'].format(question=question))
extracted = extract_final_answer(raw)

print("QUESTION :", question)
print("REFERENCE:", repr(ref))
print("RAW OUTPUT:", repr(raw))
print("EXTRACTED :", repr(extracted))
print("MATCH?    :", soft_match(extracted, ref))

In [ ]:
# DEBUG: inspect one raw prediction vs reference
example = list(datasets['TruthfulQA'])[0]
question = example['question']
ref = example['best_answer']

raw = generate_response(prompt_templates['baseline'].format(question=question))
extracted = extract_final_answer(raw)

print("QUESTION :", question)
print("REFERENCE:", repr(ref))
print("RAW OUTPUT:", repr(raw))
print("EXTRACTED :", repr(extracted))
print("MATCH?    :", soft_match(extracted, ref))

In [ ]:
def self_consistency_experiment(dataset, num_samples: int = 5) -> dict:
    preds, refs = [], []

    for i, example in enumerate(dataset):
        question = example["question"]
        ref      = example["best_answer"]

        cot_prompt = prompt_templates["cot"].format(question=question)

        raw_samples = generate_response(
            cot_prompt,
            num_samples=num_samples,
            temperature=0.7,
        )
        if isinstance(raw_samples, str):
            raw_samples = [raw_samples]

        extracted  = [extract_final_answer(s) for s in raw_samples]
        final_pred = max(set(extracted), key=extracted.count)

        preds.append(final_pred)
        refs.append(ref.lower().strip())

        if (i + 1) % 5 == 0:
            print(f"  self_consistency: {i+1}/{len(dataset)}")

    return compute_metrics(preds, refs)

print("self_consistency_experiment defined.")

In [ ]:
import os
import pandas as pd

tier_results = []

# Choose a manageable sample size per tier
POPQA_TIER_SAMPLE_SIZE = 50  # adjust down if still too slow

for tier in ["low", "mid", "high"]:
    tier_ds = popqa.filter(lambda ex: ex["popularity_tier"] == tier)
    print(f"\n=== PopQA tier: {tier}, full n={len(tier_ds)} ===")

    if len(tier_ds) == 0:
        print(f"Skipping tier {tier}: no examples.")
        continue

    # Limit to a random subset for speed
    # (datasets.Select doesn't support random directly, so we can just take first N;
    # if you want random, pre-shuffle indices.)
    tier_ds = tier_ds.select(range(min(POPQA_TIER_SAMPLE_SIZE, len(tier_ds))))
    print(f"Using {len(tier_ds)} examples for tier {tier}")

    # Baseline and CoT
    for condition in ["baseline", "cot"]:
        preds, refs, raws = [], [], []
        template = prompt_templates[condition]

        for i, ex in enumerate(tier_ds):
            if (i + 1) % 5 == 0 or i == 0:
                print(f"{tier} – {condition}: {i+1}/{len(tier_ds)}")

            question = ex["question"]
            ref = ex["best_answer"]

            prompt = template.format(question=ex["question"])
            raw_pred = generate_response(prompt)
            pred = extract_final_answer(raw_pred)

            preds.append(pred)
            refs.append(ex["best_answer"].lower().strip())
            raws.append(raw_pred.lower())

        metrics = compute_metrics(preds, refs, raw_outputs=raws)
        tier_results.append({
            "dataset": "PopQA",
            "popularity_tier": tier,
            "condition": condition,
            **metrics,
        })
        print(f"{tier} – {condition} metrics:", metrics)

    # Self-consistency
    print(f"{tier} – self_consistency running…")
    metrics_sc = self_consistency_experiment(tier_ds, num_samples=3)   # ← now works
    tier_results.append({
        "dataset": "PopQA",
        "popularity_tier": tier,
        "condition": "self_consistency",
        **metrics_sc,
    })
    print(f"{tier} – self_consistency: {metrics_sc}")

df_tier = pd.DataFrame(tier_results)
print("\nPopQA tier metrics summary:")
print(df_tier)

os.makedirs("output", exist_ok=True)
df_tier.to_csv("output/popqa_tier_results.csv", index=False)

In [ ]:
#-BREAK-

In [ ]:
# Add to conditions: Run 5 samples, majority vote
def self_consistency_experiment(dataset, num_samples: int = 5) -> dict:
    """
    Run chain-of-thought prompting with majority-vote self-consistency.

    Args:
        dataset:     HuggingFace Dataset with 'question' and 'best_answer' fields.
        num_samples: Number of independent samples to draw per question.

    Returns:
        dict with accuracy, hallucination_rate, and n.
    """
    preds, refs = [], []

    for i, example in enumerate(dataset):
        question = example["question"]
        ref      = example["best_answer"]

        cot_prompt = prompt_templates["cot"].format(question=question)

        # Draw num_samples independent responses
        raw_samples = generate_response(
            cot_prompt,
            num_samples=num_samples,
            temperature=0.7,
        )
        # generate_response returns a list when num_samples > 1
        if isinstance(raw_samples, str):
            raw_samples = [raw_samples]

        # Extract a concise answer from each sample
        extracted = [extract_final_answer(s) for s in raw_samples]

        # Majority vote (most common extracted answer)
        final_pred = max(set(extracted), key=extracted.count)

        preds.append(final_pred)
        refs.append([r.lower().strip() for r in ref] if isinstance(ref, list) else ref.lower().strip())

        if (i + 1) % 5 == 0:
            print(f"  self_consistency: {i+1}/{len(dataset)}")

    return compute_metrics(preds, refs)


print("self_consistency_experiment defined.")

# Run and append
import os
import pandas as pd

results = []
conditions = ["baseline", "cot", "self_consistency"]

for ds_name, dataset in datasets.items():
    for condition in conditions:
        print(f"\n>>> {ds_name} — {condition} ({len(dataset)} examples)")

        if condition == "self_consistency":
            # Handled by its own function
            metrics = self_consistency_experiment(dataset, num_samples=5)
            results.append({"dataset": ds_name, "condition": condition, **metrics})
            print(f"    metrics: {metrics}")
            continue

        # Baseline or CoT
        preds, refs, raws = [], [], []          # ← initialize all three
        template = prompt_templates[condition]

        for i, example in enumerate(dataset):
            if i % 50 == 0:
                print(f"  {ds_name}-{condition}: {i}/{len(dataset)}")

            question = example["question"]
            ref      = example["best_answer"]

            prompt   = template.format(question=question)
            raw_pred = generate_response(prompt)          # single response
            pred     = extract_final_answer(raw_pred)     # ← was missing

            preds.append(pred)
            #refs.append(ref.lower().strip())
            # Handle both list (TruthfulQA) and string (PopQA)
            refs.append([r.lower().strip() for r in ref] if isinstance(ref, list) else ref.lower().strip())
            raws.append(raw_pred.lower())       # ← track raw output

        metrics = compute_metrics(preds, refs, raw_outputs=raws)
        results.append({"dataset": ds_name, "condition": condition, **metrics})
        print(f"    metrics: {metrics}")

df_results = pd.DataFrame(results)
print("\n=== Results Summary ===")
print(df_results)

os.makedirs("output", exist_ok=True)
df_results.to_csv("output/results_summary.csv", index=False)


In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=df_results, x="condition", y="hallucination_rate", hue="dataset")
plt.title("Hallucination Rates by Condition and Dataset")
plt.ylabel("Hallucination Rate")
plt.ylim(0,1)
plt.savefig("hallucination_rates.png")
plt.show()

# Entity popularity analysis (for PopQA rare/common)
print("Full results saved to df_results. Export to slides!")
print(df_results.pivot(index="dataset", columns="condition", values="hallucination_rate"))

In [ ]:
# Log low-accuracy examples
low_acc = df_results[df_results['accuracy'] < 0.5]  # Customize
print("High hallucination conditions:", low_acc)

# Export to presentation-friendly CSV
df_results.to_csv("output/results_summary.csv", index=False)
print("Notebook complete! Check CSVs and PNG for slides.[file:1]")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Log low-accuracy conditions (the "Hallucination Trap")
low_acc = df_results[df_results['accuracy'] < 0.5]
print("High hallucination conditions found in:\n", low_acc)

# 2. Export to CSV for your supplementary data
os.makedirs("output", exist_ok=True)
df_results.to_csv("output/results_summary.csv", index=False)

# 3. Generate the Publication-Quality Visualization
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Create the grouped bar chart
ax = sns.barplot(
    data=df_results,
    x="dataset",
    y="accuracy",
    hue="condition",
    palette="magma"  # High-contrast palette for clear divergence
)

# Labeling for "High-Tier" presentation
plt.title("The Consistency Paradox: Accuracy Across Scaling Conditions", fontsize=14, fontweight='bold')
plt.ylabel("Accuracy Score", fontsize=12)
plt.xlabel("Dataset", fontsize=12)
plt.ylim(0, 1.0)

# Add numeric labels to bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)

plt.legend(title="Prompting Condition", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# 4. Save and Finish
plt.savefig("output/accuracy_divergence.png", dpi=300)
print("\nNotebook complete! Check the 'output' folder for results_summary.csv and accuracy_divergence.png.")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare the data for plotting
plot_df = df_results.copy()

# Set the visual style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Create the grouped bar chart
ax = sns.barplot(
    data=plot_df,
    x="dataset",
    y="accuracy",
    hue="condition",
    palette="viridis"
)

# Add title and labels
plt.title("The Consistency Paradox: Accuracy vs. Prompting Complexity", fontsize=15, pad=20)
plt.ylabel("Accuracy Score (Higher is Better)", fontsize=12)
plt.xlabel("Dataset", fontsize=12)
plt.ylim(0, 1.0) # Accuracy is a 0-1 scale

# Add values on top of bars for clarity
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)

plt.legend(title="Condition", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Save for your publication/slides
plt.savefig("output/accuracy_divergence_plot.png", dpi=300)
plt.show()


In [ ]:
!pip install transformer_lens
from transformer_lens import HookedTransformer
import torch

# Load a model that fits in your Colab RAM (e.g., GPT-2 or Llama-7B-HF)
model = HookedTransformer.from_pretrained("gpt2-small", device="cuda")


In [ ]:
#This is the Gpt2 dataset
prompt = "What happens if you swallow gum?"
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens) #Let's run with cache

# Decode the 'residual stream' at each layer
for layer in range(model.cfg.n_layers):
    layer_resid = cache["resid_post", layer]
    # Project the internal state to the vocabulary
    unembedded = model.unembed(model.ln_final(layer_resid))
    # Get the top predicted token at the final position
    top_token = unembedded[0, -1].argmax().item()
    print(f"Layer {layer}: {model.to_string(top_token)}")


In [ ]:
!pip install circuitsvis
import circuitsvis as cv

# Visualize how much each 'head' attends to specific words
str_tokens = model.to_str_tokens(prompt)
cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", 0])


In [ ]:
import torch
from transformer_lens import HookedTransformer

# Load the model in half-precision (float16) to fit on a T4 or L4 GPU
model = HookedTransformer.from_pretrained( "meta-llama/Meta-Llama-3-8B-Instruct",
    device="cuda",
    dtype=torch.float16
)

# Test prompt for the TruthfulQA "Cliff"
prompt = "What happens if you swallow gum?"
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens)

print(f"{'Layer':<10} | {'Top Prediction':<20}")
print("-" * 35)

for layer in range(model.cfg.n_layers):
    # resid_post is the state of the "thought" after each layer
    layer_resid = cache["resid_post", layer]

    # Project internal state to vocabulary
    unembedded = model.unembed(model.ln_final(layer_resid))
    top_token = unembedded[0, -1].argmax().item()

    print(f"Layer {layer:<3} | {model.to_string(top_token)}")

In [ ]:
# Test prompt for the PopQA "Stability Test"
prompt = "What is the occupation of George Rankin?" # PopQA obscure fact
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens)

print(f"{'Layer':<10} | {'Top Prediction':<20}")
print("-" * 35)

for layer in range(model.cfg.n_layers):
    layer_resid = cache["resid_post", layer]
    unembedded = model.unembed(model.ln_final(layer_resid))
    top_token = unembedded[0, -1].argmax().item()
    print(f"Layer {layer:<3} | {model.to_string(top_token)}")


In [ ]:
# We focus on Layer 21 since that's where "myth" first emerged
layer_to_probe = 21

# Get the attention patterns for that layer
# pattern shape: [batch, head, query_pos, key_pos]
attention_pattern = cache["pattern", layer_to_probe][0]

print(f"Analyzing Layer {layer_to_probe} Attention Heads...")

# Let's find which head is most active on the final token
for head in range(model.cfg.n_heads):
    # Look at the final token's attention to the rest of the prompt
    final_token_attention = attention_pattern[head, -1, :]
    max_att_val = final_token_attention.max().item()
    max_att_ind = final_token_attention.argmax().item()

    # We want heads that are looking at the "content" (gum/swallow)
    # rather than just the previous word or the start of the sentence.
    if max_att_val > 0.2: # Significant attention threshold
        source_token = model.to_string(tokens[0, max_att_ind])
        print(f"Head {head:2} | Max Attention: {max_att_val:.2f} on token: '{source_token}'")


In [ ]:
# Filter out the "Attention Sink" at index 0
for head in range(model.cfg.n_heads):
    final_token_attention = attention_pattern[head, 1 , :] #Change the head

    # We ignore index 0 (the sink) and look at the rest
    non_sink_attention = final_token_attention[1:]
    max_val = non_sink_attention.max().item()
    max_ind = non_sink_attention.argmax().item() + 1 # +1 because we sliced

    if max_val > 0.05: # Lower threshold to find the subtle truth
        source_token = model.to_string(tokens[0, max_ind])
        print(f"Head {head:2} | Sub-Signal: {max_val:.2f} on token: '{source_token}'")


In [ ]:
#THE FIX

for head in range(model.cfg.n_heads):
    # Final token's attention to all positions, excluding BOS sink
    final_token_attention = attention_pattern[head, -1, 1:]  # slice directly
    max_val = final_token_attention.max().item()
    max_ind = final_token_attention.argmax().item() + 1  # +1 to correct for slice

    if max_val > 0.05:
        source_token = model.to_string(tokens[0, max_ind])
        print(f"Head {head:2d} | Sub-Signal: {max_val:.2f} on token: '{source_token}'")